In [ ]:

!pip -q install torch torchvision pandas datasets scipy matplotlib tqdm

from __future__ import annotations

import copy
import gc
import json
import math
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib import pyplot as plt
from scipy import stats
from torch.func import functional_call
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm

try:
    import google.colab  
    RUN_IN_COLAB = True
except Exception:
    RUN_IN_COLAB = False

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


In [ ]:
@dataclass(frozen=True)
class Config:
    dataset_id: str = "uoft-cs/cifar10"
    boundary_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    continuation_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)
    train_per_class: int = 1000
    test_per_class: int = 500
    batch_size: int = 128
    boundary_epochs: int = 12
    boundary_lr: float = 1e-3
    response_lrs: Tuple[float, ...] = (3e-4, 1e-3)
    beta1: float = 0.9
    beta2: float = 0.999
    adam_eps: float = 1e-8
    weight_decay: float = 0.0
    horizons: Tuple[int, ...] = (1, 5, 10, 20, 40)
    memory_components: Tuple[str, ...] = ("m", "rms_global", "rms_layerwise", "rms_random_layerwise")
    m_finite_difference_scales: Tuple[float, ...] = (1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2)
    rms_finite_difference_scales: Tuple[float, ...] = (1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
    n_directions: int = 4
    response_rank: int = 8
    horizon_diagnostic_lrs: Tuple[float, ...] = (3e-5, 1e-4, 3e-4, 1e-3)
    run_horizon_diagnostic: bool = True
    paper_primary_lr: float = 3e-4
    paper_primary_horizon: int = 20
    paper_primary_scale: float = 1e-2
    seed: int = 0
    num_workers: int = 0
    run_spectrum: bool = False
    force: bool = False

CFG = Config(
    seed=int(os.environ.get("ADAM_RTTP_SEED", "0")),
    run_spectrum=os.environ.get("ADAM_RTTP_RUN_SPECTRUM", "0") != "0",
    run_horizon_diagnostic=os.environ.get("ADAM_RTTP_RUN_HORIZON_DIAGNOSTIC", "1") != "0",
    force=os.environ.get("ADAM_RTTP_FORCE", "0") == "1",
)

USE_GOOGLE_DRIVE = os.environ.get("ADAM_RTTP_USE_GOOGLE_DRIVE", "1") != "0"
DRIVE_BASE = Path(os.environ.get("ADAM_RTTP_DRIVE_BASE", "/content/drive/MyDrive"))
ROOT_NAME = os.environ.get("ADAM_RTTP_ROOT_NAME", "adam_paper_aligned_rms_response_v2")

if RUN_IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = DRIVE_BASE / ROOT_NAME
else:
    ROOT = Path.cwd() / ROOT_NAME

TAB = ROOT / "tables"
FIG = ROOT / "figures"
CACHE = ROOT / "cache"
CHUNKS = CACHE / "chunks"
for p in (ROOT, TAB, FIG, CACHE, CHUNKS):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print("device:", DEVICE)
print("root:", ROOT)
print("config:", CFG)
(ROOT / "config.json").write_text(json.dumps(asdict(CFG), indent=2))


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def format_duration(seconds: float) -> str:
    seconds = max(0, int(round(float(seconds))))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"


class EtaTimer:
    def __init__(self, total: int, label: str):
        self.total = max(int(total), 1)
        self.label = label
        self.start = time.perf_counter()
        self.done = 0

    def update(self, n: int = 1, note: str = ""):
        self.done += int(n)
        elapsed = time.perf_counter() - self.start
        rate = self.done / elapsed if elapsed > 0 else float("nan")
        remaining = (self.total - self.done) / rate if np.isfinite(rate) and rate > 0 else float("nan")
        eta = format_duration(remaining) if np.isfinite(remaining) else "unknown"
        suffix = f" | {note}" if note else ""
        print(f"[{self.label}] {self.done}/{self.total} | elapsed {format_duration(elapsed)} | ETA {eta}{suffix}")


def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def atomic_torch_save(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


seed_everything(CFG.seed)


COMPONENT_SEED = {"m": 101, "rms_global": 211, "rms_layerwise": 307, "rms_random_layerwise": 401}


In [ ]:
class HFCifarTask(Dataset):
    def __init__(self, hf_split, classes: Sequence[int], per_class: int, train: bool):
        self.classes = tuple(int(c) for c in classes)
        self.label_map = {c: i for i, c in enumerate(self.classes)}
        counts = {c: 0 for c in self.classes}
        rows = []
        for row in hf_split:
            y = int(row["label"])
            if y in counts and counts[y] < per_class:
                rows.append(row)
                counts[y] += 1
            if all(v >= per_class for v in counts.values()):
                break
        missing = {c: per_class - n for c, n in counts.items() if n < per_class}
        if missing:
            raise ValueError(f"not enough examples per class: {missing}")
        self.rows = rows
        self.transform = (
            v2.Compose([
                v2.ToImage(),
                v2.RandomCrop(32, padding=4),
                v2.RandomHorizontalFlip(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
            ])
            if train else
            v2.Compose([
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
            ])
        )

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        return self.transform(row["img"]), self.label_map[int(row["label"])]


def make_data(seed: int):
    seed_everything(seed)
    ds = load_dataset(CFG.dataset_id)
    A_tr = HFCifarTask(ds["train"], CFG.boundary_classes, CFG.train_per_class, True)
    B_tr = HFCifarTask(ds["train"], CFG.continuation_classes, CFG.train_per_class, True)
    A_te = HFCifarTask(ds["test"], CFG.boundary_classes, CFG.test_per_class, False)
    B_te = HFCifarTask(ds["test"], CFG.continuation_classes, CFG.test_per_class, False)
    kw = dict(batch_size=CFG.batch_size, num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available())
    return {
        "A_train": DataLoader(A_tr, shuffle=True, generator=torch.Generator().manual_seed(seed), **kw),
        "B_train": DataLoader(B_tr, shuffle=True, generator=torch.Generator().manual_seed(seed + 10_000), **kw),
        "A_test": DataLoader(A_te, shuffle=False, **kw),
        "B_test": DataLoader(B_te, shuffle=False, **kw),
    }


def materialize_batches(loader: DataLoader, H: int):
    out = []
    it = iter(loader)
    while len(out) < H:
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader)
            x, y = next(it)
        out.append((x.to(device=DEVICE, dtype=DTYPE), y.to(DEVICE)))
    return out


data = make_data(CFG.seed)
future_batches = materialize_batches(data["B_train"], max(CFG.horizons))
print("Task A classes:", CFG.boundary_classes)
print("Task B classes:", CFG.continuation_classes)
print("future batches:", len(future_batches))


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, n_classes)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))


model_template = SmallCNN(n_classes=5).to(device=DEVICE, dtype=DTYPE)
param_items = list(model_template.named_parameters())
param_names = [name for name, _ in param_items]
param_shapes = [(p.shape, p.numel()) for _, p in param_items]

PARAM_NAMES = param_names
PARAM_SPECS = param_shapes
NUM_PARAMS = int(sum(n for _, n in param_shapes))


def flatten_model(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])


def vector_to_param_dict(theta: torch.Tensor) -> Dict[str, torch.Tensor]:
    out = {}
    offset = 0
    for name, (shape, n) in zip(param_names, param_shapes):
        out[name] = theta[offset:offset+n].view(shape)
        offset += n
    assert offset == theta.numel()
    return out


def batch_loss(theta: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> torch.Tensor:
    logits = functional_call(model_template, vector_to_param_dict(theta), (xb,))
    return F.cross_entropy(logits, yb)


print("SmallCNN parameters:", NUM_PARAMS)


In [ ]:
@dataclass
class AdamState:
    theta: torch.Tensor
    m: torch.Tensor
    v: torch.Tensor
    step: int


@dataclass
class AdamTangent:
    dtheta: torch.Tensor
    dm: torch.Tensor
    dv: torch.Tensor


def clone_adam_state(s: AdamState) -> AdamState:
    return AdamState(s.theta.detach().clone(), s.m.detach().clone(), s.v.detach().clone(), int(s.step))


def adam_denominator_and_dsqrt(v_hat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    floor = CFG.adam_eps ** 2
    active = v_hat > floor
    sqrt_v = torch.sqrt(torch.clamp(v_hat, min=floor))
    dsqrt_dv = torch.where(active, 0.5 / sqrt_v, torch.zeros_like(v_hat))
    return sqrt_v + CFG.adam_eps, dsqrt_dv


def grad_only(theta: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    theta_req = theta.detach().requires_grad_(True)
    loss = batch_loss(theta_req, xb, yb)
    g = torch.autograd.grad(loss, theta_req)[0]
    return loss.detach(), g.detach()


def grad_and_hvp(theta: torch.Tensor, vec: torch.Tensor, xb: torch.Tensor, yb: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    theta_req = theta.detach().requires_grad_(True)
    loss = batch_loss(theta_req, xb, yb)
    g = torch.autograd.grad(loss, theta_req, create_graph=True)[0]
    hvp = torch.autograd.grad(torch.dot(g, vec), theta_req, retain_graph=False)[0]
    return g.detach(), hvp.detach()


def adam_step(state: AdamState, xb: torch.Tensor, yb: torch.Tensor, lr: float) -> Tuple[AdamState, float]:
    loss, g = grad_only(state.theta, xb, yb)
    step = state.step + 1
    b1, b2 = CFG.beta1, CFG.beta2
    m_next = b1 * state.m + (1 - b1) * g
    v_next = b2 * state.v + (1 - b2) * g.square()
    c1 = 1 - b1 ** step
    c2 = 1 - b2 ** step
    m_hat = m_next / c1
    v_hat = v_next / c2
    denom, _ = adam_denominator_and_dsqrt(v_hat)
    theta_next = state.theta - lr * m_hat / denom
    return AdamState(theta_next.detach(), m_next.detach(), v_next.detach(), step), float(loss)


def adam_tangent_step(
    nominal: AdamState,
    tangent: AdamTangent,
    xb: torch.Tensor,
    yb: torch.Tensor,
    lr: float,
    restricted_no_dv: bool = False,
) -> Tuple[AdamState, AdamTangent]:
    g, h_dtheta = grad_and_hvp(nominal.theta, tangent.dtheta, xb, yb)
    step = nominal.step + 1
    b1, b2 = CFG.beta1, CFG.beta2

    m_next = b1 * nominal.m + (1 - b1) * g
    v_next = b2 * nominal.v + (1 - b2) * g.square()

    dm_next = b1 * tangent.dm + (1 - b1) * h_dtheta
    if restricted_no_dv:
        dv_next = torch.zeros_like(tangent.dv)
    else:
        dv_next = b2 * tangent.dv + 2 * (1 - b2) * g * h_dtheta

    c1 = 1 - b1 ** step
    c2 = 1 - b2 ** step
    m_hat = m_next / c1
    v_hat = v_next / c2
    dm_hat = dm_next / c1
    dv_hat = dv_next / c2
    denom, dsqrt_dv = adam_denominator_and_dsqrt(v_hat)

    d_update = dm_hat / denom - m_hat * dsqrt_dv * dv_hat / denom.square()
    dtheta_next = tangent.dtheta - lr * d_update
    theta_next = nominal.theta - lr * m_hat / denom

    if not torch.isfinite(dtheta_next).all():
        raise FloatingPointError("non-finite Adam tangent")

    return (
        AdamState(theta_next.detach(), m_next.detach(), v_next.detach(), step),
        AdamTangent(dtheta_next.detach(), dm_next.detach(), dv_next.detach()),
    )


In [ ]:
def train_or_load_boundary() -> Tuple[AdamState, pd.DataFrame]:
    path = CACHE / f"boundary_adam_smallcnn_seed{CFG.seed}.pt"
    if path.exists() and not CFG.force:
        obj = torch.load(path, map_location=DEVICE)
        state = AdamState(obj["theta"].to(DEVICE), obj["m"].to(DEVICE), obj["v"].to(DEVICE), int(obj["step"]))
        losses = pd.DataFrame(obj.get("losses", []))
        print("loaded boundary:", path)
        return state, losses

    seed_everything(CFG.seed)
    theta0 = flatten_model(model_template).to(device=DEVICE, dtype=DTYPE)
    state = AdamState(theta0, torch.zeros_like(theta0), torch.zeros_like(theta0), 0)
    rows = []
    timer = EtaTimer(CFG.boundary_epochs * len(data["A_train"]), "boundary Adam Task A")
    for epoch in range(CFG.boundary_epochs):
        for xb, yb in data["A_train"]:
            xb = xb.to(device=DEVICE, dtype=DTYPE)
            yb = yb.to(DEVICE)
            state, loss = adam_step(state, xb, yb, CFG.boundary_lr)
            rows.append(dict(epoch=epoch, step=state.step, loss=loss))
            if state.step % 50 == 0:
                timer.update(n=50, note=f"step={state.step} loss={loss:.3f}")
    losses = pd.DataFrame(rows)
    atomic_torch_save({
        "theta": state.theta.detach().cpu(),
        "m": state.m.detach().cpu(),
        "v": state.v.detach().cpu(),
        "step": int(state.step),
        "losses": rows,
        "config": asdict(CFG),
    }, path)
    atomic_write_csv(losses, TAB / "boundary_losses.csv")
    print("saved boundary:", path)
    return state, losses


boundary, boundary_losses = train_or_load_boundary()
print("boundary step:", boundary.step)
print("||theta||:", float(boundary.theta.norm()))
print("||m||:", float(boundary.m.norm()))
print("||v||:", float(boundary.v.norm()))
print("min v:", float(boundary.v.min()), "zero v count:", int((boundary.v == 0).sum()))


In [ ]:

def nominal_adam_state(boundary: AdamState) -> AdamState:
    
    
    return AdamState(
        boundary.theta.detach().clone(),
        boundary.m.detach().clone(),
        boundary.v.detach().clone(),
        int(boundary.step),
    )


def parameter_slices() -> Dict[str, slice]:
    names = globals().get("PARAM_NAMES", param_names)
    specs = globals().get("PARAM_SPECS", param_shapes)
    out = {}
    offset = 0
    for name, (_, n) in zip(names, specs):
        out[name] = slice(offset, offset + n)
        offset += n
    return out


PARAM_SLICES = parameter_slices()
PARAMETER_TENSORS = list(PARAM_SLICES.keys())


def component_dim(component: str) -> int:
    return NUM_PARAMS


def component_direction_count(component: str) -> int:
    if component == "rms_global":
        return 1
    if component == "rms_layerwise":
        return len(PARAMETER_TENSORS)
    return CFG.n_directions


def component_radius_base(component: str, base: AdamState) -> float:
    if component == "m":
        return max(float(base.m.norm()), 1e-12)
    
    return 1.0


def component_fd_scales(component: str) -> Tuple[float, ...]:
    if component == "m":
        return CFG.m_finite_difference_scales
    return CFG.rms_finite_difference_scales


def random_unit_direction(dim: int, seed: int) -> torch.Tensor:
    gen = torch.Generator(device="cpu").manual_seed(int(seed))
    z = torch.randn(dim, generator=gen, dtype=DTYPE).to(DEVICE)
    return z / z.norm().clamp_min(1e-30)


def rms_direction(component: str, direction_id: int, seed: int) -> Tuple[torch.Tensor, str]:
    if component == "rms_global":
        return torch.ones(NUM_PARAMS, device=DEVICE, dtype=DTYPE), "all_parameters"

    if component == "rms_layerwise":
        name = PARAMETER_TENSORS[int(direction_id)]
        q = torch.zeros(NUM_PARAMS, device=DEVICE, dtype=DTYPE)
        q[PARAM_SLICES[name]] = 1.0
        return q, name

    if component == "rms_random_layerwise":
        gen = torch.Generator(device="cpu").manual_seed(int(seed))
        coeffs = torch.randn(len(PARAMETER_TENSORS), generator=gen, dtype=DTYPE)
        coeffs = coeffs / coeffs.norm().clamp_min(1e-30)
        q = torch.zeros(NUM_PARAMS, device=DEVICE, dtype=DTYPE)
        for coefficient, name in zip(coeffs, PARAMETER_TENSORS):
            q[PARAM_SLICES[name]] = coefficient.to(DEVICE)
        return q, f"random_layerwise_{direction_id}"

    raise ValueError(component)


def component_coordinate(
    component: str,
    direction_id: int,
    base: AdamState,
) -> Tuple[torch.Tensor, str]:
    seed = CFG.seed + 100_000 + 1000 * int(direction_id) + COMPONENT_SEED[component]

    if component == "m":
        unit = random_unit_direction(NUM_PARAMS, seed)
        coord = component_radius_base(component, base) * unit
        return coord, f"random_m_{direction_id}"

    return rms_direction(component, direction_id, seed)


def coordinate_tangent(
    base: AdamState,
    component: str,
    coord: torch.Tensor,
) -> AdamTangent:
    z = torch.zeros(NUM_PARAMS, device=DEVICE, dtype=DTYPE)

    if component == "m":
        return AdamTangent(z.clone(), coord.clone(), z.clone())

    if component.startswith("rms_"):
        
        return AdamTangent(z.clone(), z.clone(), 2.0 * base.v * coord)

    raise ValueError(component)


def state_from_coordinate(
    base: AdamState,
    component: str,
    coord: torch.Tensor,
    eps_scale: float,
) -> AdamState:
    if component == "m":
        return AdamState(
            base.theta.clone(),
            base.m + float(eps_scale) * coord,
            base.v.clone(),
            base.step,
        )

    if component.startswith("rms_"):
        exponent = torch.clamp(
            2.0 * float(eps_scale) * coord,
            min=-20.0,
            max=20.0,
        )
        return AdamState(
            base.theta.clone(),
            base.m.clone(),
            base.v * torch.exp(exponent),
            base.step,
        )

    raise ValueError(component)


def state_parameterization(component: str) -> str:
    if component == "m":
        return "additive_first_moment"
    return "multiplicative_rms_memory"


base_state = nominal_adam_state(boundary)
print("nominal step offset:", base_state.step)
print("nominal inherited min v:", float(base_state.v.min()))
print("parameter tensors:", PARAMETER_TENSORS)


In [ ]:
def theta_snapshots_from_state(initial: AdamState, lr: float, horizons: Sequence[int]) -> Dict[int, torch.Tensor]:
    wanted = set(int(h) for h in horizons)
    state = clone_adam_state(initial)
    out = {}
    for local_step, (xb, yb) in enumerate(future_batches, start=1):
        state, _ = adam_step(state, xb, yb, lr)
        if local_step in wanted:
            out[local_step] = state.theta.detach().clone()
    return out


def tangent_snapshots(component: str, coord: torch.Tensor, lr: float, horizons: Sequence[int], restricted_no_dv: bool = False) -> Tuple[Dict[int, torch.Tensor], Dict[int, float]]:
    wanted = set(int(h) for h in horizons)
    nominal = clone_adam_state(base_state)
    tangent = coordinate_tangent(base_state, component, coord)
    theta_out = {}
    dv_norm = {0: float(tangent.dv.norm())}
    for local_step, (xb, yb) in enumerate(future_batches, start=1):
        nominal, tangent = adam_tangent_step(nominal, tangent, xb, yb, lr, restricted_no_dv=restricted_no_dv)
        dv_norm[local_step] = float(tangent.dv.norm())
        if local_step in wanted:
            theta_out[local_step] = tangent.dtheta.detach().clone()
    return theta_out, dv_norm


def rel_error(pred: torch.Tensor, true: torch.Tensor, tol: float = 1e-12) -> float:
    den = float(true.norm())
    if den < tol:
        return np.nan
    return float((pred - true).norm() / den)


def cosine(pred: torch.Tensor, true: torch.Tensor, tol: float = 1e-12) -> float:
    pn = float(pred.norm())
    tn = float(true.norm())
    if min(pn, tn) < tol:
        return np.nan
    return float(torch.dot(pred, true) / (pred.norm() * true.norm()))



def run_direction(
    component: str,
    direction_id: int,
    lr: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    coord, direction_label = component_coordinate(
        component, direction_id, base_state
    )

    full_tangent, full_dv = tangent_snapshots(
        component,
        coord,
        lr,
        CFG.horizons,
        restricted_no_dv=False,
    )

    restricted_tangent, restricted_dv = ({}, {})
    if component == "m":
        restricted_tangent, restricted_dv = tangent_snapshots(
            component,
            coord,
            lr,
            CFG.horizons,
            restricted_no_dv=True,
        )

    nominal = theta_snapshots_from_state(
        base_state, lr, CFG.horizons
    )
    rows = []

    for scale in component_fd_scales(component):
        plus_state = state_from_coordinate(
            base_state, component, coord, float(scale)
        )
        minus_state = state_from_coordinate(
            base_state, component, coord, -float(scale)
        )
        plus = theta_snapshots_from_state(
            plus_state, lr, CFG.horizons
        )
        minus = theta_snapshots_from_state(
            minus_state, lr, CFG.horizons
        )

        v_plus_nonnegative = bool(
            torch.isfinite(plus_state.v).all()
            and (plus_state.v >= 0).all()
        )
        v_minus_nonnegative = bool(
            torch.isfinite(minus_state.v).all()
            and (minus_state.v >= 0).all()
        )

        for H in CFG.horizons:
            true_endpoint = plus[H] - nominal[H]
            centered = (
                plus[H] - minus[H]
            ) / (2 * float(scale))

            for tangent_model, tangent_by_H in [
                ("full_coupled", full_tangent),
                ("restricted_no_dv", restricted_tangent),
            ]:
                if H not in tangent_by_H:
                    continue

                tan = tangent_by_H[H]
                pred_endpoint = float(scale) * tan

                rows.append(dict(
                    optimizer="adam",
                    architecture="smallcnn",
                    seed=CFG.seed,
                    direction=direction_id,
                    direction_label=direction_label,
                    learning_rate=lr,
                    beta1=CFG.beta1,
                    beta2=CFG.beta2,
                    adam_eps=CFG.adam_eps,
                    memory_component=component,
                    tangent_model=tangent_model,
                    state_parameterization=state_parameterization(component),
                    nominal_memory="inherited",
                    step_offset=base_state.step,
                    horizon=H,
                    radius_scale=float(scale),
                    radius_base=component_radius_base(component, base_state),
                    coordinate_norm=float(coord.norm()),
                    alpha=float(scale) if component.startswith("rms_") else np.nan,
                    rms_change_fraction=(
                        float(scale)
                        if component in ("rms_global", "rms_layerwise")
                        else np.nan
                    ),
                    max_abs_log_rms_change=(
                        float(scale) * float(coord.abs().max())
                        if component.startswith("rms_")
                        else np.nan
                    ),
                    rms_log_change_l2_layer_space=(
                        float(scale)
                        if component == "rms_random_layerwise"
                        else np.nan
                    ),
                    rms_direction_nonzero=int((coord != 0).sum().item()),
                    tangent_norm=float(tan.norm()),
                    true_endpoint_norm=float(true_endpoint.norm()),
                    centered_fd_norm=float(centered.norm()),
                    endpoint_error=rel_error(
                        pred_endpoint, true_endpoint
                    ),
                    endpoint_cosine=cosine(
                        pred_endpoint, true_endpoint
                    ),
                    centered_error=rel_error(tan, centered),
                    centered_cosine=cosine(tan, centered),
                    v_plus_nonnegative=v_plus_nonnegative,
                    v_minus_nonnegative=v_minus_nonnegative,
                ))

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dv_rows = []
    for tangent_model, dv_hist in [
        ("full_coupled", full_dv),
        ("restricted_no_dv", restricted_dv),
    ]:
        if not dv_hist:
            continue

        for step, value in dv_hist.items():
            dv_rows.append(dict(
                optimizer="adam",
                architecture="smallcnn",
                seed=CFG.seed,
                direction=direction_id,
                direction_label=direction_label,
                learning_rate=lr,
                memory_component=component,
                tangent_model=tangent_model,
                step=step,
                dv_norm=value,
            ))

    return pd.DataFrame(rows), pd.DataFrame(dv_rows)


In [ ]:

def run_one_step_audit() -> pd.DataFrame:
    path = TAB / "adam_one_step_audit.csv"
    
    rows = []
    xb, yb = future_batches[0]

    for component in CFG.memory_components:
        
        direction_id = 0
        coord, direction_label = component_coordinate(
            component, direction_id, base_state
        )
        tangent = coordinate_tangent(base_state, component, coord)
        _, tangent_next = adam_tangent_step(
            clone_adam_state(base_state),
            tangent,
            xb,
            yb,
            CFG.response_lrs[0],
            restricted_no_dv=False,
        )

        for scale in component_fd_scales(component):
            plus, _ = adam_step(
                state_from_coordinate(
                    base_state, component, coord, float(scale)
                ),
                xb,
                yb,
                CFG.response_lrs[0],
            )
            minus, _ = adam_step(
                state_from_coordinate(
                    base_state, component, coord, -float(scale)
                ),
                xb,
                yb,
                CFG.response_lrs[0],
            )
            centered = (plus.theta - minus.theta) / (2 * float(scale))

            rows.append(dict(
                optimizer="adam",
                architecture="smallcnn",
                seed=CFG.seed,
                memory_component=component,
                direction_label=direction_label,
                learning_rate=CFG.response_lrs[0],
                radius_scale=float(scale),
                centered_error=rel_error(tangent_next.dtheta, centered),
                centered_cosine=cosine(tangent_next.dtheta, centered),
                tangent_norm=float(tangent_next.dtheta.norm()),
                centered_fd_norm=float(centered.norm()),
            ))

    out = pd.DataFrame(rows)
    atomic_write_csv(out, path)
    return out


one_step_audit = run_one_step_audit()
one_step_audit.head()


In [ ]:

def response_chunk_path(
    lr: float,
    component: str,
    direction: int,
) -> Path:
    return CHUNKS / (
        f"response_adam_smallcnn_seed={CFG.seed}_grid=component_fd_v2_"
        f"lr={lr:g}_mem={component}_dir={direction}.csv"
    )


def dv_chunk_path(
    lr: float,
    component: str,
    direction: int,
) -> Path:
    return CHUNKS / (
        f"dv_adam_smallcnn_seed={CFG.seed}_grid=component_fd_v2_"
        f"lr={lr:g}_mem={component}_dir={direction}.csv"
    )


all_rows = []
all_dv = []
total = sum(
    component_direction_count(component)
    for _ in CFG.response_lrs
    for component in CFG.memory_components
)
timer = EtaTimer(total, "Adam response grid")

for lr in CFG.response_lrs:
    for component in CFG.memory_components:
        for direction in range(
            component_direction_count(component)
        ):
            rpath = response_chunk_path(
                lr, component, direction
            )
            dpath = dv_chunk_path(
                lr, component, direction
            )

            if (
                rpath.exists()
                and dpath.exists()
                and not CFG.force
            ):
                rows = pd.read_csv(rpath)
                dv_rows = pd.read_csv(dpath)
                note = "loaded"
            else:
                rows, dv_rows = run_direction(
                    component, direction, lr
                )
                atomic_write_csv(rows, rpath)
                atomic_write_csv(dv_rows, dpath)
                note = "computed"

            all_rows.append(rows)
            all_dv.append(dv_rows)
            timer.update(
                note=(
                    f"{note} lr={lr:g} "
                    f"mem={component} dir={direction}"
                )
            )

response_raw = pd.concat(all_rows, ignore_index=True)
dv_response = pd.concat(all_dv, ignore_index=True)
atomic_write_csv(
    response_raw, TAB / "adam_response_raw.csv"
)
atomic_write_csv(
    dv_response, TAB / "adam_dv_response.csv"
)
response_raw.head()


In [ ]:

def adam_step_with_diagnostics(state: AdamState, xb: torch.Tensor, yb: torch.Tensor, lr: float) -> Tuple[AdamState, Dict[str, float]]:
    loss, g = grad_only(state.theta, xb, yb)
    step = state.step + 1
    b1, b2 = CFG.beta1, CFG.beta2
    m_next = b1 * state.m + (1 - b1) * g
    v_next = b2 * state.v + (1 - b2) * g.square()
    c1 = 1 - b1 ** step
    c2 = 1 - b2 ** step
    m_hat = m_next / c1
    v_hat = v_next / c2
    denom, _ = adam_denominator_and_dsqrt(v_hat)
    update = lr * m_hat / denom
    theta_next = state.theta - update
    diag = dict(
        loss=float(loss),
        grad_norm=float(g.norm()),
        m_norm=float(m_next.norm()),
        v_norm=float(v_next.norm()),
        v_min=float(v_next.min()),
        denom_min=float(denom.min()),
        denom_median=float(denom.median()),
        denom_max=float(denom.max()),
        update_norm=float(update.norm()),
        theta_norm=float(theta_next.norm()),
        update_to_theta=float(update.norm() / theta_next.norm().clamp_min(1e-30)),
    )
    return AdamState(theta_next.detach(), m_next.detach(), v_next.detach(), step), diag


def nominal_horizon_diagnostics(lr: float) -> pd.DataFrame:
    state = clone_adam_state(base_state)
    theta0 = base_state.theta.detach().clone()
    rows = []
    wanted = set(CFG.horizons)
    for local_step, (xb, yb) in enumerate(future_batches, start=1):
        state, diag = adam_step_with_diagnostics(state, xb, yb, lr)
        if local_step in wanted:
            rows.append(dict(
                learning_rate=lr,
                horizon=local_step,
                theta_displacement_norm=float((state.theta - theta0).norm()),
                **diag,
            ))
    return pd.DataFrame(rows)


def representative_horizon_diagnostic(lr: float, component: str, direction_id: int = 0) -> Tuple[pd.DataFrame, pd.DataFrame]:
    coord, label = component_coordinate(component, direction_id, base_state)
    tangent_by_H, _ = tangent_snapshots(component, coord, lr, CFG.horizons, restricted_no_dv=False)
    nominal = theta_snapshots_from_state(base_state, lr, CFG.horizons)
    tangent_rows = []
    fd_rows = []

    for H, tan in tangent_by_H.items():
        tangent_rows.append(dict(
            learning_rate=lr,
            memory_component=component,
            direction=direction_id,
            direction_label=label,
            horizon=H,
            coordinate_norm=float(coord.norm()),
            tangent_norm=float(tan.norm()),
            tangent_gain_per_coordinate_norm=float(tan.norm() / coord.norm().clamp_min(1e-30)),
            theta_norm=float(nominal[H].norm()),
        ))

    eps_machine = torch.finfo(DTYPE).eps
    for scale in component_fd_scales(component):
        plus_state = state_from_coordinate(base_state, component, coord, float(scale))
        minus_state = state_from_coordinate(base_state, component, coord, -float(scale))
        plus = theta_snapshots_from_state(plus_state, lr, CFG.horizons)
        minus = theta_snapshots_from_state(minus_state, lr, CFG.horizons)
        for H in CFG.horizons:
            tan = tangent_by_H[H]
            endpoint = plus[H] - nominal[H]
            pred = float(scale) * tan
            centered = (plus[H] - minus[H]) / (2 * float(scale))
            fd_delta = plus[H] - minus[H]
            theta_norm = float(nominal[H].norm())
            fd_rho = float(fd_delta.norm() / (eps_machine * max(theta_norm, 1e-30)))
            residual = float((endpoint - pred).norm())
            fd_rows.append(dict(
                learning_rate=lr,
                memory_component=component,
                direction=direction_id,
                direction_label=label,
                horizon=H,
                radius_scale=float(scale),
                tangent_norm=float(tan.norm()),
                endpoint_norm=float(endpoint.norm()),
                predicted_endpoint_norm=float(pred.norm()),
                centered_fd_norm=float(centered.norm()),
                endpoint_error=rel_error(pred, endpoint),
                endpoint_cosine=cosine(pred, endpoint),
                centered_error=rel_error(tan, centered),
                centered_cosine=cosine(tan, centered),
                first_order_residual=residual,
                residual_over_endpoint=float(residual / endpoint.norm().clamp_min(1e-30)),
                fd_delta_norm=float(fd_delta.norm()),
                fd_rho=fd_rho,
                fd_resolvable_1e3=bool(fd_rho > 1e3),
            ))
    return pd.DataFrame(tangent_rows), pd.DataFrame(fd_rows)


def run_horizon_failure_diagnostics() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    traj_path = TAB / "adam_horizon_nominal_diagnostics.csv"
    tan_path = TAB / "adam_horizon_tangent_growth.csv"
    fd_path = TAB / "adam_horizon_fd_diagnostics.csv"
    summary_path = TAB / "adam_horizon_failure_summary.csv"
    if all(path.exists() for path in (traj_path, tan_path, fd_path, summary_path)) and not CFG.force:
        return pd.read_csv(traj_path), pd.read_csv(tan_path), pd.read_csv(fd_path), pd.read_csv(summary_path)

    components = list(CFG.memory_components)
    trajectory_rows = []
    tangent_rows = []
    fd_rows = []
    total = len(CFG.horizon_diagnostic_lrs) * (1 + len(components))
    timer = EtaTimer(total, "Adam H20 diagnostic")

    for lr in CFG.horizon_diagnostic_lrs:
        trajectory_rows.append(nominal_horizon_diagnostics(float(lr)))
        timer.update(note=f"nominal lr={lr:g}")
        for component in components:
            tan_df, fd_df = representative_horizon_diagnostic(float(lr), component, direction_id=0)
            tangent_rows.append(tan_df)
            fd_rows.append(fd_df)
            timer.update(note=f"lr={lr:g} mem={component}")

    traj = pd.concat(trajectory_rows, ignore_index=True)
    tan = pd.concat(tangent_rows, ignore_index=True)
    fd = pd.concat(fd_rows, ignore_index=True)

    best_rows = []
    for keys, g in fd.groupby(["learning_rate", "memory_component", "horizon"]):
        lr, component, H = keys
        finite = g[np.isfinite(g.centered_error)].copy()
        if len(finite):
            best = finite.loc[finite.centered_error.idxmin()].to_dict()
            best["selection"] = "min_centered_error_scale"
            best_rows.append(best)
    summary = pd.DataFrame(best_rows)
    if len(summary):
        summary["passes_local_025_cos09"] = (summary.centered_error < 0.25) & (summary.centered_cosine > 0.9)
        max_pass = []
        for keys, g in summary.groupby(["learning_rate", "memory_component"]):
            lr, component = keys
            passed = g[g.passes_local_025_cos09]
            h20 = g[g.horizon == 20]
            max_pass.append(dict(
                learning_rate=lr,
                memory_component=component,
                max_horizon_passed=int(passed.horizon.max()) if len(passed) else 0,
                best_H20_centered_error=float(h20.centered_error.iloc[0]) if len(h20) else np.nan,
                best_H20_centered_cosine=float(h20.centered_cosine.iloc[0]) if len(h20) else np.nan,
            ))
        summary = summary.merge(pd.DataFrame(max_pass), on=["learning_rate", "memory_component"], how="left")

    atomic_write_csv(traj, traj_path)
    atomic_write_csv(tan, tan_path)
    atomic_write_csv(fd, fd_path)
    atomic_write_csv(summary, summary_path)
    return traj, tan, fd, summary


if CFG.run_horizon_diagnostic:
    horizon_nominal_diagnostics, horizon_tangent_growth, horizon_fd_diagnostics, horizon_failure_summary = run_horizon_failure_diagnostics()
else:
    horizon_nominal_diagnostics = pd.DataFrame()
    horizon_tangent_growth = pd.DataFrame()
    horizon_fd_diagnostics = pd.DataFrame()
    horizon_failure_summary = pd.DataFrame()

horizon_failure_summary.head()


In [ ]:

def spectrum_from_columns(columns: List[torch.Tensor]) -> Dict[str, float]:
    r = len(columns)
    if r == 0:
        return dict(
            response_rank=0,
            frobenius_norm=np.nan,
            spectral_norm=np.nan,
            stable_rank=np.nan,
            effective_rank=np.nan,
            energy_top1=np.nan,
            energy_top2=np.nan,
            rank90=np.nan,
        )
    G = torch.empty((r, r), dtype=torch.float64)
    for i in range(r):
        for j in range(r):
            G[i, j] = float(torch.dot(columns[i].reshape(-1), columns[j].reshape(-1)))
    G = torch.nan_to_num((G + G.T) / 2, nan=0.0, posinf=0.0, neginf=0.0)
    trace = float(torch.trace(G).abs())
    eye = torch.eye(r, dtype=torch.float64)
    last_error = None
    for jitter in (0.0, 1e-14, 1e-12, 1e-10):
        try:
            eig = torch.linalg.eigvalsh(G + jitter * max(trace, 1.0) * eye).clamp_min(0)
            break
        except RuntimeError as exc:
            last_error = exc
    else:
        raise last_error
    sing = torch.sqrt(eig).flip(0).cpu().numpy()
    energy = sing ** 2
    total = float(energy.sum())
    probs = energy / total if total > 0 else np.zeros_like(energy)
    entropy = float(-(probs[probs > 0] * np.log(probs[probs > 0])).sum()) if total > 0 else np.nan
    cum = np.cumsum(probs) if total > 0 else np.zeros_like(probs)
    row = dict(
        response_rank=r,
        frobenius_norm=math.sqrt(total),
        spectral_norm=float(sing[0]) if len(sing) else np.nan,
        stable_rank=float(total / (sing[0] ** 2)) if len(sing) and sing[0] > 0 else np.nan,
        effective_rank=math.exp(entropy) if np.isfinite(entropy) else np.nan,
        energy_top1=float(probs[0]) if len(probs) else np.nan,
        energy_top2=float(probs[:2].sum()) if len(probs) else np.nan,
        rank90=int(np.searchsorted(cum, 0.9) + 1) if total > 0 else np.nan,
    )
    for k, s in enumerate(sing, start=1):
        row[f"singular_value_{k}"] = float(s)
    return row


def spectrum_direction_count(component: str) -> int:
    if component == "rms_global":
        return 1
    if component == "rms_layerwise":
        return len(PARAMETER_TENSORS)
    return CFG.response_rank


def spectrum_chunk_path(lr: float, component: str) -> Path:
    return CHUNKS / (
        f"spectrum_adam_smallcnn_seed={CFG.seed}_"
        f"lr={lr:g}_mem={component}.csv"
    )


def coordinate_basis_description(component: str) -> str:
    if component == "m":
        return "random Euclidean first-moment probes scaled by ||m0||"
    if component == "rms_global":
        return "single global multiplicative RMS-memory percentage coordinate"
    if component == "rms_layerwise":
        return "one multiplicative RMS-memory percentage coordinate per parameter tensor"
    if component == "rms_random_layerwise":
        return "random coefficients normalized over parameter tensors, expanded to RMS-memory coordinates"
    raise ValueError(component)


def run_spectrum_component(lr: float, component: str) -> pd.DataFrame:
    path = spectrum_chunk_path(lr, component)
    if path.exists() and not CFG.force:
        return pd.read_csv(path)

    count = spectrum_direction_count(component)
    rows = []
    columns_by_H = {H: [] for H in CFG.horizons}
    labels = []
    coord_norms = []
    nonzeros = []

    for direction_id in tqdm(range(count), desc=f"spectrum {component} lr={lr:g}"):
        coord, label = component_coordinate(component, direction_id, base_state)
        snaps, _ = tangent_snapshots(component, coord, lr, CFG.horizons, restricted_no_dv=False)
        labels.append(label)
        coord_norms.append(float(coord.norm()))
        nonzeros.append(int((coord != 0).sum().item()))
        for H in CFG.horizons:
            columns_by_H[H].append(snaps[H])

    for H in CFG.horizons:
        row = spectrum_from_columns(columns_by_H[H])
        row.update(dict(
            optimizer="adam",
            architecture="smallcnn",
            seed=CFG.seed,
            learning_rate=lr,
            beta1=CFG.beta1,
            beta2=CFG.beta2,
            memory_component=component,
            state_parameterization=state_parameterization(component),
            nominal_memory="inherited",
            step_offset=base_state.step,
            horizon=H,
            coordinate_basis=coordinate_basis_description(component),
        rank_note="no effective-rank claim without an explicit input metric",
            coordinate_count=count,
            mean_coordinate_norm=float(np.mean(coord_norms)) if coord_norms else np.nan,
            max_coordinate_norm=float(np.max(coord_norms)) if coord_norms else np.nan,
            mean_nonzero_coordinates=float(np.mean(nonzeros)) if nonzeros else np.nan,
            spectrum_object="optional coordinate-Gram diagnostic for selected, non-orthonormal natural coordinates R_H U",
        ))
        rows.append(row)

    out = pd.DataFrame(rows)
    atomic_write_csv(out, path)
    return out


if CFG.run_spectrum:
    spectra = []
    for lr in CFG.response_lrs:
        for component in CFG.memory_components:
            spectra.append(run_spectrum_component(lr, component))
    response_spectrum = pd.concat(spectra, ignore_index=True)
else:
    response_spectrum = pd.DataFrame()
    print("Spectrum diagnostic skipped by default; set ADAM_RTTP_RUN_SPECTRUM=1 to compute coordinate-Gram diagnostics.")
atomic_write_csv(response_spectrum, TAB / "adam_response_spectrum.csv")

storage_rows = []
float_bytes = 4
for component in CFG.memory_components:
    k = spectrum_direction_count(component)
    storage_rows.append(dict(
        optimizer="adam",
        architecture="smallcnn",
        seed=CFG.seed,
        memory_component=component,
        coordinate_basis=coordinate_basis_description(component),
        rank_note="no effective-rank claim without an explicit input metric",
        parameter_count=NUM_PARAMS,
        dense_single_block_response_bytes=NUM_PARAMS * NUM_PARAMS * float_bytes,
        reduced_response_columns=k,
        reduced_response_bytes=NUM_PARAMS * k * float_bytes,
        tangent_state_bytes_per_direction=3 * NUM_PARAMS * float_bytes,
        compression_vs_dense_block=(NUM_PARAMS * NUM_PARAMS) / max(NUM_PARAMS * k, 1),
        basis_storage="implicit for RMS global/layerwise; explicit random vector seeds for m/random-layerwise",
    ))
storage_accounting = pd.DataFrame(storage_rows)
atomic_write_csv(storage_accounting, TAB / "adam_storage_accounting.csv")
response_spectrum.head()


In [ ]:

def ci95(x):
    x = pd.Series(x).dropna().to_numpy(float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    m = float(x.mean())
    if len(x) == 1:
        return m, np.nan, np.nan
    se = float(x.std(ddof=1) / math.sqrt(len(x)))
    q = float(stats.t.ppf(0.975, len(x)-1))
    return m, m - q * se, m + q * se


def mean_ci_columns(g: pd.DataFrame, cols: Sequence[str]) -> Dict[str, float]:
    out = {}
    for col in cols:
        m, lo, hi = ci95(g[col])
        out[col] = m
        out[col + "_lo"] = lo
        out[col + "_hi"] = hi
    return out


summary_cols = [
    "endpoint_error",
    "centered_error",
    "endpoint_cosine",
    "centered_cosine",
    "tangent_norm",
    "true_endpoint_norm",
    "centered_fd_norm",
]
summary_rows = []
seed_level_rows = []

for keys, seed_g in response_raw.groupby([
    "seed", "learning_rate", "memory_component", "tangent_model", "horizon", "radius_scale"
]):
    seed, lr, component, tangent_model, H, scale = keys
    seed_row = dict(
        seed=seed,
        learning_rate=lr,
        memory_component=component,
        tangent_model=tangent_model,
        horizon=H,
        radius_scale=scale,
        n_directions=len(seed_g),
    )
    for col in summary_cols:
        seed_row[col] = float(seed_g[col].mean())
    seed_level_rows.append(seed_row)

seed_level_response = pd.DataFrame(seed_level_rows)
atomic_write_csv(seed_level_response, TAB / "adam_seed_level_response.csv")

for keys, g in seed_level_response.groupby([
    "learning_rate", "memory_component", "tangent_model", "horizon", "radius_scale"
]):
    lr, component, tangent_model, H, scale = keys
    row = dict(
        learning_rate=lr,
        memory_component=component,
        tangent_model=tangent_model,
        horizon=H,
        radius_scale=scale,
        n_seeds=g.seed.nunique(),
        n_seed_direction_means=len(g),
    )
    row.update(mean_ci_columns(g, summary_cols))
    summary_rows.append(row)

response_summary_by_scale = pd.DataFrame(summary_rows)
atomic_write_csv(response_summary_by_scale, TAB / "adam_response_summary_by_scale.csv")

best_rows = []
for keys, g in response_summary_by_scale.groupby([
    "learning_rate", "memory_component", "tangent_model", "horizon"
]):
    finite = g[np.isfinite(g.centered_error)].copy()
    if len(finite):
        best = finite.loc[finite.centered_error.idxmin()].to_dict()
        best["selection"] = "min_centered_error_scale"
        best_rows.append(best)
adam_best_scale_summary = pd.DataFrame(best_rows)
atomic_write_csv(adam_best_scale_summary, TAB / "adam_best_scale_summary.csv")

paper_best = adam_best_scale_summary[
    adam_best_scale_summary.tangent_model == "full_coupled"
].copy()
atomic_write_csv(
    paper_best,
    TAB / "paper_aligned_adam_rms_full_coupled_best_scales.csv",
)

paper_table = paper_best[
    (paper_best.learning_rate == CFG.paper_primary_lr)
    & (paper_best.horizon.isin(CFG.horizons))
].copy()
if len(paper_table):
    paper_table = paper_table[[
        "learning_rate",
        "memory_component",
        "horizon",
        "radius_scale",
        "centered_error",
        "centered_cosine",
        "endpoint_error",
        "endpoint_cosine",
        "tangent_norm",
        "true_endpoint_norm",
        "n_seeds",
    ]].sort_values(["memory_component", "horizon"])
atomic_write_csv(paper_table, TAB / "paper_table_support_adam_rms_smallcnn.csv")

paper_compact = paper_table[
    paper_table.horizon == CFG.paper_primary_horizon
].copy() if len(paper_table) else pd.DataFrame()
atomic_write_csv(paper_compact, TAB / "paper_table_support_adam_rms_smallcnn_compact.csv")


layer_rows = []
layer_df = response_raw[
    (response_raw.tangent_model == "full_coupled")
    & (response_raw.memory_component == "rms_layerwise")
    & (response_raw.learning_rate == CFG.paper_primary_lr)
    & (response_raw.horizon == CFG.paper_primary_horizon)
].copy()
for keys, g in layer_df.groupby(["direction", "direction_label"]):
    direction, label = keys
    finite = g[np.isfinite(g.centered_error)].copy()
    if len(finite):
        best = finite.loc[finite.centered_error.idxmin()].to_dict()
        layer_rows.append(best)
paper_layerwise = pd.DataFrame(layer_rows)
if len(paper_layerwise):
    paper_layerwise = paper_layerwise[[
        "direction",
        "direction_label",
        "radius_scale",
        "centered_error",
        "centered_cosine",
        "endpoint_error",
        "endpoint_cosine",
        "tangent_norm",
        "true_endpoint_norm",
        "coordinate_norm",
    ]].sort_values("direction")
atomic_write_csv(paper_layerwise, TAB / "paper_table_support_adam_rms_layerwise_directions.csv")


def compact_tex_table(df: pd.DataFrame) -> str:
    if not len(df):
        return "% No rows available. Run the response grid first.\n"
    lines = [
        "\\begin{tabular}{lrrrrr}",
        "\\toprule",
        'Memory & H & scale & centered err. & centered cos. & endpoint err. \\\\',
        "\\midrule",
    ]
    for _, r in df.iterrows():
        lines.append(
            f"{r['memory_component']} & {int(r['horizon'])} & {r['radius_scale']:.3g} & "
            f"{r['centered_error']:.3g} & {r['centered_cosine']:.3f} & {r['endpoint_error']:.3g} "
            '\\\\'
        )
    lines.extend(["\\bottomrule", "\\end{tabular}"])
    return "\n".join(lines) + "\n"

(ROOT / "tables" / "paper_table_support_adam_rms_smallcnn.tex").write_text(compact_tex_table(paper_compact))

claim_rows = []
def add_claim(claim, status, evidence, paper_use):
    claim_rows.append(dict(claim=claim, status=status, evidence=evidence, paper_use=paper_use))

add_claim(
    "Adam first-moment response",
    "supported if compact table shows low centered error and high cosine",
    "full coupled m tangent; restricted-no-dv diagnostic retained separately",
    "supporting evidence for general optimizer-state response",
)
add_claim(
    "Adam RMS global response",
    "supported if finite differences agree across the local alpha sweep",
    "global multiplicative RMS coordinate q=1, v0(alpha)=v0 exp(2 alpha)",
    "supporting evidence for natural second-moment intervention",
)
add_claim(
    "Adam RMS layerwise response",
    "supported direction-wise, not as a dense full log-v spectrum",
    "one multiplicative RMS coordinate per parameter tensor",
    "supporting evidence for interpretable reduced second-moment subspaces",
)
add_claim(
    "Dense log-v unit-vector response",
    "not claimed here",
    "previous audit found the derivative correct but low-gain/FD-limited",
    "diagnostic only",
)
claim_support = pd.DataFrame(claim_rows)
atomic_write_csv(claim_support, TAB / "adam_claim_support_status.csv")

quality = []
def add_check(check, passed, detail, blocking=True):
    quality.append(dict(check=check, passed=bool(passed), blocking=bool(blocking), detail=str(detail)))

add_check(
    "one_step_audit_finite",
    bool(np.isfinite(one_step_audit.centered_error).all()),
    one_step_audit.centered_error.describe().to_dict(),
)
add_check(
    "response_raw_finite",
    bool(np.isfinite(response_raw[["endpoint_error", "centered_error", "endpoint_cosine", "centered_cosine"]].to_numpy()).all()),
    response_raw.isna().sum().to_dict(),
)
add_check(
    "v_perturbations_nonnegative",
    bool(response_raw.v_plus_nonnegative.all() and response_raw.v_minus_nonnegative.all()),
    "all plus/minus v states nonnegative",
)
add_check(
    "paper_table_created",
    bool(len(paper_compact) > 0),
    f"compact rows={len(paper_compact)} at lr={CFG.paper_primary_lr:g}, H={CFG.paper_primary_horizon}",
    blocking=False,
)

add_check(
    "horizon_failure_diagnostic_created",
    bool((not CFG.run_horizon_diagnostic) or len(horizon_failure_summary) > 0),
    f"diagnostic rows={len(horizon_failure_summary) if CFG.run_horizon_diagnostic else 0}; run_horizon_diagnostic={CFG.run_horizon_diagnostic}",
    blocking=False,
)

if len(response_spectrum):
    add_check(
        "optional_coordinate_gram_spectrum_finite",
        bool(np.isfinite(response_spectrum[["frobenius_norm", "spectral_norm", "effective_rank"]].to_numpy()).all()),
        "diagnostic only; not a paper effective-rank claim without an explicit input metric",
        blocking=False,
    )

for (lr, comp), g in adam_best_scale_summary[(adam_best_scale_summary.tangent_model == "full_coupled")].groupby(["learning_rate", "memory_component"]):
    h = g[g.horizon == CFG.paper_primary_horizon]
    if len(h):
        err = float(h.centered_error.iloc[0])
        cosv = float(h.centered_cosine.iloc[0])
        add_check(
            f"paper_horizon_local_agreement_lr={lr:g}_mem={comp}",
            bool((err < 0.25) and (cosv > 0.9)),
            f"best centered error={err:.3g}, cosine={cosv:.3g} at H={CFG.paper_primary_horizon}",
            blocking=False,
        )

if "restricted_no_dv" in response_raw.tangent_model.unique():
    m_full = adam_best_scale_summary[(adam_best_scale_summary.memory_component == "m") & (adam_best_scale_summary.tangent_model == "full_coupled")]
    m_rest = adam_best_scale_summary[(adam_best_scale_summary.memory_component == "m") & (adam_best_scale_summary.tangent_model == "restricted_no_dv")]
    merged = m_full.merge(m_rest, on=["learning_rate", "memory_component", "horizon"], suffixes=("_full", "_restricted"))
    if len(merged):
        diff = (merged.centered_error_restricted - merged.centered_error_full).abs().mean()
        add_check(
            "full_vs_restricted_m_difference_reported",
            True,
            f"mean abs centered-error gap is {diff:.3g}; table retained for interpretation",
            blocking=False,
        )
        atomic_write_csv(merged, TAB / "paper_aligned_adam_m_full_vs_restricted.csv")

quality_report = pd.DataFrame(quality)
atomic_write_csv(quality_report, TAB / "adam_quality_report.csv")
quality_report


In [ ]:

def savefig(name):
    plt.tight_layout()
    plt.savefig(
        FIG / f"{name}.png",
        dpi=220,
        bbox_inches="tight",
    )
    plt.savefig(
        FIG / f"{name}.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


plot_df = response_summary_by_scale[
    (response_summary_by_scale.tangent_model == "full_coupled")
    & (response_summary_by_scale.horizon == CFG.paper_primary_horizon)
]

plt.figure(figsize=(7.5, 4.8))
for (lr, comp), g in plot_df.groupby(["learning_rate", "memory_component"]):
    g = g.sort_values("radius_scale")
    plt.loglog(
        g.radius_scale,
        g.centered_error,
        marker="o",
        label=f"{comp}, lr={lr:g}",
    )
plt.xlabel("perturbation scale alpha")
plt.ylabel("centered derivative relative error")
plt.title(f"Adam full-coupled response locality, H={CFG.paper_primary_horizon}")
plt.legend(frameon=False)
savefig("paper_adam_rms_locality_primary_horizon")


horizon_df = adam_best_scale_summary[
    (adam_best_scale_summary.tangent_model == "full_coupled")
    & (adam_best_scale_summary.learning_rate == CFG.paper_primary_lr)
]
if len(horizon_df):
    plt.figure(figsize=(7.5, 4.8))
    for comp, g in horizon_df.groupby("memory_component"):
        g = g.sort_values("horizon")
        plt.semilogy(g.horizon, g.centered_error, marker="o", label=comp)
    plt.xlabel("horizon H")
    plt.ylabel("best centered derivative relative error")
    plt.title(f"Adam prediction quality across horizons, lr={CFG.paper_primary_lr:g}")
    plt.legend(frameon=False)
    savefig("paper_adam_rms_prediction_by_horizon")


m_df = response_summary_by_scale[
    response_summary_by_scale.memory_component == "m"
]
if len(m_df):
    plt.figure(figsize=(7.5, 4.8))
    for (lr, model_name), g in m_df[
        m_df.horizon == CFG.paper_primary_horizon
    ].groupby(["learning_rate", "tangent_model"]):
        g = g.sort_values("radius_scale")
        plt.loglog(
            g.radius_scale,
            g.centered_error,
            marker="o",
            label=f"{model_name}, lr={lr:g}",
        )
    plt.xlabel("perturbation scale")
    plt.ylabel("centered derivative relative error")
    plt.title(f"Adam m-response: full coupled vs restricted, H={CFG.paper_primary_horizon}")
    plt.legend(frameon=False)
    savefig("paper_adam_m_full_vs_restricted_primary_horizon")


rms_df = response_summary_by_scale[
    response_summary_by_scale.memory_component.str.startswith("rms_")
    & (response_summary_by_scale.tangent_model == "full_coupled")
    & (response_summary_by_scale.horizon == CFG.paper_primary_horizon)
    & (response_summary_by_scale.radius_scale == CFG.paper_primary_scale)
]
if len(rms_df):
    plt.figure(figsize=(8.5, 4.8))
    labels = [
        f"{c}\nlr={lr:g}"
        for c, lr in zip(
            rms_df.memory_component,
            rms_df.learning_rate,
        )
    ]
    plt.bar(labels, rms_df.true_endpoint_norm)
    plt.ylabel("mean nonlinear parameter displacement norm")
    plt.title(f"Natural RMS-memory interventions at alpha={CFG.paper_primary_scale:g}, H={CFG.paper_primary_horizon}")
    plt.xticks(rotation=25, ha="right")
    savefig("paper_adam_rms_response_magnitude_primary")


if len(response_spectrum):
    plt.figure(figsize=(7.5, 4.8))
    for (lr, comp), g in response_spectrum.groupby(["learning_rate", "memory_component"]):
        g = g.sort_values("horizon")
        plt.plot(g.horizon, g.effective_rank, marker="o", label=f"{comp}, lr={lr:g}")
    plt.xlabel("horizon H")
    plt.ylabel("effective rank of selected coordinate Gram")
    plt.title("Adam optional coordinate-Gram diagnostic")
    plt.legend(frameon=False)
    savefig("diagnostic_adam_rms_coordinate_gram_effective_rank")


if len(paper_layerwise):
    layer_plot = paper_layerwise.sort_values("tangent_norm", ascending=False)
    plt.figure(figsize=(8.5, 4.8))
    plt.bar(layer_plot.direction_label, layer_plot.tangent_norm)
    plt.ylabel("||R_H q_layer||")
    plt.title(f"Layerwise RMS-memory response magnitude, H={CFG.paper_primary_horizon}")
    plt.xticks(rotation=35, ha="right")
    savefig("paper_adam_rms_layerwise_response_magnitude")


In [ ]:

manifest = {
    "notebook": "adam_paper_aligned_rms_response_v2.ipynb",
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "config": asdict(CFG),
    "scientific_object": (
        "finite-horizon Adam optimizer-memory response for additive first-moment "
        "perturbations and natural global/layerwise multiplicative RMS-memory perturbations"
    ),
    "paper_scope": (
        "supporting evidence for optimizer-state generality and Adam state decomposition; "
        "compact Adam table targets H=20, with separate diagnostics to explain horizon failures; "
        "heavy-ball SGD remains the main expository and frozen-baseline experiment"
    ),
    "rms_boundary_map": (
        "r0(alpha) = r0 * exp(alpha q); equivalently "
        "v0(alpha) = v0 * exp(2 alpha q); dv0/dalpha = 2 v0 * q"
    ),
    "rms_direction_families": [
        "global percentage scaling",
        "one percentage-scaling direction per parameter tensor",
        "random coefficients normalized in layer space",
    ],
    "adam_denominator": "sqrt(clamp(v_hat, adam_eps^2)) + adam_eps",
    "boundary_state": "exact inherited Adam m, raw v, and step counter after Task A; no pre-continuation clamp of stored v",
    "continuation": (
        "Task B materialized minibatch sequence held fixed for tangent and finite differences"
    ),
    "paper_tables": [
        "paper_aligned_adam_rms_full_coupled_best_scales.csv",
        "paper_table_support_adam_rms_smallcnn.csv",
        "paper_table_support_adam_rms_smallcnn_compact.csv",
        "paper_table_support_adam_rms_layerwise_directions.csv",
        "paper_table_support_adam_rms_smallcnn.tex",
        "adam_storage_accounting.csv",
        "adam_claim_support_status.csv",
    ],
    "diagnostic_tables": [
        "adam_response_raw.csv",
        "adam_response_summary_by_scale.csv",
        "adam_best_scale_summary.csv",
        "adam_response_spectrum.csv (empty unless ADAM_RTTP_RUN_SPECTRUM=1; diagnostic only)",
        "adam_dv_response.csv",
        "adam_one_step_audit.csv",
        "adam_quality_report.csv",
        "adam_horizon_nominal_diagnostics.csv",
        "adam_horizon_tangent_growth.csv",
        "adam_horizon_fd_diagnostics.csv",
        "adam_horizon_failure_summary.csv",
    ],
}
(ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = ROOT / "adam_paper_aligned_rms_response_v2_artifacts.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (TAB, FIG):
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=str(path.relative_to(ROOT)))
    zf.write(ROOT / "config.json", arcname="config.json")
    zf.write(ROOT / "manifest.json", arcname="manifest.json")

print("archive:", zip_path)
